# Figure 3: normal-system spectra and theoretical bounds

Run all cells to generate `results/figure_03.png` and `results/figure_03.csv`.
A 300-iteration GKS reconstruction initializes the MM and IAS PS-GKS solves.
The spectra are plotted at weighted iterations 0, 20, and 50.


In [ ]:
print("Device: CPU")

from pathlib import Path
import sys

# Launch Jupyter from the repository root.
ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

import contextlib
import numpy as np
import pandas as pd
import scipy.sparse as sps
from scipy.fft import dct, idct
import matplotlib.pyplot as plt
import operators
from threadpoolctl import threadpool_limits

# Small projected solves are faster and more repeatable with one BLAS thread.
_blas_threads = threadpool_limits(limits=1)

# Figure color palette.
colors = [(106,192,205), (235,186,63), (172,46,55), (32,82,145),
          (83,165,106), (239,145,75), (208,92,89), (80,67,157),
          (82,82,82), (53,53,53), (30,27,28)]
plt.rc("axes", prop_cycle=plt.cycler(color=[tuple(v/255 for v in c) for c in colors]))


## Synthetic data and first-difference prior

The triangular first-difference inverse is applied by cumulative sums.


In [ ]:
# Synthetic piecewise-constant signal and the first 50 orthonormal DCT coefficients.
n = 1000
ground_truth = np.empty(n)
ground_truth[:250] = 2.5
ground_truth[250:400] = -0.5
ground_truth[400:625] = 4.5
ground_truth[625:825] = 2.5
ground_truth[825:] = 0.5

class DCT1D(operators._CustomLinearOperator):
    def __init__(self, n):
        super().__init__(
            (n, n),
            lambda x: dct(np.squeeze(x), type=2, norm="ortho"),
            lambda x: idct(np.squeeze(x), type=2, norm="ortho"),
            device="cpu", dtype=None,
        )

mask = np.zeros(n)
mask[:50] = 1
A = operators.Subsampling1DOperator(mask) @ DCT1D(n)
b_true = A @ ground_truth
# RandomState uses the same random-number sequence as np.random.seed(0).
noise = np.random.RandomState(0).normal(size=A.shape[0])
noise /= np.linalg.norm(noise)
noise *= 0.03 * np.linalg.norm(b_true)
b = b_true + noise
noise_var = (0.03 * np.linalg.norm(b_true))**2 / A.shape[0]

Lmat, _ = operators.first_order_derivative_1d(n, boundary="zero")
L = operators.MatrixLinearOperator(Lmat)
# The inverse of L = I - upper-shift is applied by cumulative sums.
from experiment_1d import FirstDifferenceInverse
Lpinv = FirstDifferenceInverse(n)
Lpinv_factory = lambda w: Lpinv @ operators.MatrixLinearOperator(sps.diags(1.0/w))


## Run the two plotted PS-GKS methods

MM uses $p=1$, $\epsilon=10^{-3}$; IAS uses $r=-1$, $\beta=1$. The discrepancy principle uses $\eta=1.01$. Full solver messages go to `results/figure_03_solver.log`.

In [ ]:
from msolver import VanillaGKS, MMXGKS, IASGKS

solver_arguments = dict(noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=None)
solve_arguments = dict(init_dim=5, lambdah_min=1e-7, lambdah_max=1e7,
                       regparam="dp", x_true=ground_truth, ksparsity=None,
                       regparam_kwargs={"eta": 1.01})
time_steps = [0, 20, 50]

with (RESULTS / "figure_03_solver.log").open("w") as log, \
        contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
    baseline = VanillaGKS(A, L, b, **solver_arguments).solve(
        n_iter=300, x0=None, lambdah0=1.0, weight_params={}, xker=None,
        **solve_arguments)
    mm_solver = MMXGKS(A, L, b, **solver_arguments)
    ias_solver = IASGKS(A, L, b, **solver_arguments)
    histories = {}
    for family, solver, weights in [
        ("MM", mm_solver, {"q": 1.0}),
        ("IAS", ias_solver, {"hyperpr_r": -1.0, "hyperpr_beta": 1.0}),
    ]:
        histories[family] = solver.solve_with_priorconditioning(
            n_iter=max(time_steps), x0=baseline["x"], lambdah0=baseline["lambdah"],
            weight_params=weights, keep_solution_history=True, **solve_arguments)

# The solver whitens A by the noise standard deviation; the paper spectra use this A.
Amat = operators.black_box_to_dense(mm_solver.A)
print("Computed MM/IAS histories through iteration", max(time_steps))


## Compute every plotted eigenvalue and bound

Rows in the CSV identify the weight family, iteration, and zero-based eigenvalue index. Spectra are sorted in descending order. Numerical eigenvalues are clipped below at $10^{-16}$.

In [ ]:
def descending_eigenvalues(matrix):
    return np.maximum(np.linalg.eigvalsh(matrix)[::-1], 1e-16)

# These matrices are independent of the weight family and iteration.
AtA = Amat.T @ Amat
AtA_eig = descending_eigenvalues(AtA)
LtL_eig = descending_eigenvalues((Lmat.T @ Lmat).toarray())
Lmatinv = np.triu(np.ones((n, n)))
ALinv = Amat @ Lmatinv
spectrum_frames = []
for family, history in histories.items():
    for iteration in time_steps:
        _, weights, mu = history["solution_history"][iteration][:3]
        weighted_L = sps.diags(weights) @ Lmat
        Qst = AtA + mu * (weighted_L.T @ weighted_L).toarray()
        transformed_A = ALinv / weights[None, :]
        Qpr = mu * np.eye(n) + transformed_A.T @ transformed_A
        Wsq_eig = np.sort(weights**2)[::-1]
        Winvsq_eig = np.sort(1.0 / weights**2)[::-1]
        bound_one = mu + AtA_eig * Winvsq_eig[0] / LtL_eig[-1]
        bound_two = mu + AtA_eig[0] * Winvsq_eig / LtL_eig[-1]
        spectrum_frames.append(pd.DataFrame({
            "weights": family, "iteration": iteration, "eigenvalue_index": np.arange(n),
            "mu": mu,
            "standard_eigenvalue": descending_eigenvalues(Qst),
            "priorconditioned_eigenvalue": descending_eigenvalues(Qpr),
            "standard_lower_bound": AtA_eig[-1] + mu*LtL_eig[-1]*Wsq_eig,
            "standard_upper_bound": AtA_eig[0] + mu*LtL_eig[0]*Wsq_eig,
            "priorconditioned_lower_bound": np.full(n, mu),
            "priorconditioned_upper_bound": np.minimum(bound_one, bound_two),
            "upper_bound_one": bound_one, "upper_bound_two": bound_two,
        }))

spectrum = pd.concat(spectrum_frames, ignore_index=True)
spectrum.to_csv(RESULTS / "figure_03.csv", index=False)
spectrum.head()


## Plot Figure 3

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 10))
for row, family in enumerate(["MM", "IAS"]):
    for column, iteration in enumerate(time_steps):
        frame = spectrum[(spectrum.weights == family) & (spectrum.iteration == iteration)]
        ax = axs[row, column]
        index = frame.eigenvalue_index.to_numpy()
        ax.fill_between(index, frame.standard_lower_bound, frame.standard_upper_bound,
                        label=r"Theoretical bounds for $\lambda_i(\mathbf{Q}_{\mu}^{\mathrm{st}})$",
                        color="C3", alpha=0.5)
        ax.fill_between(index, frame.priorconditioned_lower_bound, frame.priorconditioned_upper_bound,
                        label=r"Theoretical bounds for $\lambda_i(\mathbf{Q}_{\mu}^{\mathrm{pr}})$",
                        color="C2", alpha=0.5)
        ax.semilogy(index, frame.standard_eigenvalue,
                    label=r"Realized $\lambda_i(\mathbf{Q}_{\mu}^{\mathrm{st}})$", color="C3", lw=3)
        ax.semilogy(index, frame.priorconditioned_eigenvalue,
                    label=r"Realized $\lambda_i(\mathbf{Q}_{\mu}^{\mathrm{pr}})$", color="C2", lw=3)
        ax.semilogy(index, frame.upper_bound_one,
                    label=r"$\mu + c_1 \lambda_i(\mathbf{A}^T \mathbf{A})$", ls=":", color="C6", lw=2)
        ax.semilogy(index, frame.upper_bound_two,
                    label=r"$\mu + c_2 \lambda_i(\mathbf{W}^{-2})$", ls=":", color="C4", lw=2)
        ax.legend(loc="upper right")
        ax.set_title(f"PS-GKS with {family} weights, iteration {iteration}")
        ax.set_xlabel("Eigenvalue index")
        ax.set_ylim(1e-4, 1e8 if family == "MM" else 1e16)
fig.tight_layout()
fig.savefig(RESULTS / "figure_03.png", dpi=250)
plt.show()
